# NoSQL en Databricks · 01 · Modelado documental
**Objetivo:** entender documentos, anidamiento, referencias y esquemas flexibles.
Duración: 30 minutos. Ejecutá primero 00 y conservá `student_id` y escala.
Cada notebook carga su propia configuración; las variables Python no se comparten entre notebooks.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "test", ["test", "small"], "02 - Escala NoSQL")

In [ ]:
%run ../common/config

In [ ]:
config = CourseConfig(
    spark.sql("SELECT current_catalog()").first()[0],
    dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"),
)
repetitions = {"test": 1, "small": 100}[config.scale]
required = ["nosql_orders", "nosql_raw_orders", "nosql_quarantine",
            "nosql_customers", "nosql_products"]
missing = [name for name in required
           if not spark.catalog.tableExists(qualified_table(config, name))]
assert not missing, f"Faltan {missing}. Ejecutá 00_setup_datos con el mismo student_id."
assert spark.table(qualified_table(config, "nosql_orders")).count() == 12 * repetitions,             "La escala no coincide con 00_setup_datos."
spark.sql(f"USE CATALOG {quote_identifier(config.catalog)}")
spark.sql(f"USE SCHEMA {quote_identifier(config.schema)}")
print(f"Laboratorio: {config.catalog}.{config.schema}; escala {config.scale}")

## 1. Elegir el agregado
Un pedido contiene los datos que necesitamos leer juntos. `items` es una lista de objetos;
`customer_snapshot` es la copia histórica del cliente en el momento de la compra.
`customer_id` y `items[].product_id` permiten consultar entidades actuales por referencia.

Antes de ejecutar: dibujá una versión relacional con pedidos, detalle, clientes y productos.
¿Qué joins evita el pedido documental? ¿Qué información se repite?

In [ ]:
%sql
SELECT order_id, doc:customer_snapshot AS customer_snapshot,
       doc:shipping AS shipping, doc:items AS items,
       doc:items[0].product_id::STRING AS first_product
FROM nosql_orders ORDER BY order_id LIMIT 12

## 2. Snapshot embebido y referencia
La misma entidad puede representarse de las dos formas por motivos distintos.
El snapshot responde cómo era el pedido; la referencia permite ver la entidad actual.
El generador usa productos y clientes independientes de las otras prácticas.

In [ ]:
%sql
SELECT o.order_id, o.doc:customer_id::STRING AS customer_id,
       o.doc:customer_snapshot.name::STRING AS name_at_purchase,
       c.doc:name::STRING AS current_name,
       o.doc:items[0].unit_price::DECIMAL(12,2) AS price_at_purchase,
       p.doc:current_price::DECIMAL(12,2) AS current_price
FROM nosql_orders AS o
LEFT JOIN nosql_customers AS c ON o.doc:customer_id::STRING = c.customer_id
LEFT JOIN nosql_products AS p ON o.doc:items[0].product_id::STRING = p.product_id
ORDER BY o.order_id LIMIT 12

Para `O000001`, el precio histórico de la notebook es USD 1.000 y el actual es USD 1.100.
**No recalcules la venta histórica con el precio actual.**
El pedido sin líneas (`O000011`) permanece gracias al LEFT JOIN; su primer producto es ausente.
¿Cuándo embebés una dirección? ¿Conviene embeber todos los pedidos dentro del cliente?
Considerá crecimiento sin límite, patrones de lectura y frecuencia de actualización.

## 3. Documentos de versiones distintas en una sola columna
La versión 2 agrega `loyalty_level`, `device` y cupones opcionales sin cambiar el esquema físico
de la tabla. `schema_version` es un atributo definido por la aplicación.

In [ ]:
%sql
SELECT order_id, doc:schema_version::INT AS version,
       doc:customer_snapshot.loyalty_level::STRING AS loyalty,
       doc:device.platform::STRING AS platform,
       schema_of_variant(doc:total) AS original_total_type,
       try_variant_get(doc, '$.total', 'decimal(12,2)') AS total
FROM nosql_orders ORDER BY order_id LIMIT 12

In [ ]:
%sql
SELECT doc:schema_version::INT AS version, COUNT(*) AS orders,
       schema_of_variant_agg(doc) AS observed_schema
FROM nosql_orders GROUP BY doc:schema_version::INT ORDER BY version

Extraemos tipos escalares antes de agrupar: `VARIANT` tiene restricciones de comparación y agrupación.
En el pedido 12 el total es texto numérico; `try_variant_get` permite convertirlo al contrato decimal.
La flexibilidad conserva los campos, pero una consulta debe conocer su significado y validar tipos.

## 4. Ausente, null explícito y texto vacío
Antes de castear, distinguí los valores de `coupon`. Una extracción a STRING puede ocultar la diferencia
entre ausencia y null. No reemplaces todos estos casos automáticamente por el mismo valor.

In [ ]:
%sql
SELECT order_id, doc:coupon AS raw_coupon,
       CASE WHEN doc:coupon IS NULL THEN 'ausente'
            WHEN is_variant_null(doc:coupon) THEN 'null_explicito'
            WHEN try_variant_get(doc, '$.coupon', 'string') = '' THEN 'texto_vacio'
            ELSE 'con_valor' END AS coupon_state
FROM nosql_orders
WHERE doc:schema_version::INT = 2
ORDER BY order_id LIMIT 4

In [ ]:
states = spark.sql("""
SELECT order_id,
       CASE WHEN doc:coupon IS NULL THEN 'ausente'
            WHEN is_variant_null(doc:coupon) THEN 'null_explicito'
            WHEN try_variant_get(doc, '$.coupon', 'string') = '' THEN 'texto_vacio'
            ELSE 'con_valor' END AS state
FROM nosql_orders WHERE order_id IN ('O000009','O000010','O000011','O000012')
""").collect()
assert {r.order_id: r.state for r in states} == {
    "O000009": "ausente", "O000010": "null_explicito",
    "O000011": "texto_vacio", "O000012": "con_valor"}

## 5. Proyectar un contrato fijo con PySpark
`from_json` con un STRUCT explícito conserva los campos declarados para esa proyección.
El documento VARIANT sigue guardando los demás. Observá que no declaramos `loyalty_level`.
Ambas representaciones sirven para necesidades distintas.

In [ ]:
from pyspark.sql import functions as F
typed_schema = "order_id STRING, customer_snapshot STRUCT<name:STRING,country:STRING>"
comparison = (spark.table("nosql_orders")
              .withColumn("fixed", F.from_json("raw_json", typed_schema))
              .select("order_id", "fixed.customer_snapshot",
                      F.expr("doc:customer_snapshot.loyalty_level::STRING").alias("extra_field")))
display(comparison.orderBy("order_id").limit(12))
comparison.printSchema()

### Decisiones de diseño para entregar
1. Justificá el snapshot del cliente y el precio del producto embebido en cada línea.
2. ¿Qué cambiarías si se requiere editar el correo del cliente sin tocar pedidos históricos?
3. ¿Qué consulta se simplifica con este modelo y cuál puede hacerse más costosa?
4. ¿Por qué el esquema flexible no elimina los contratos, la calidad ni las migraciones de aplicación?
5. ¿Qué perdemos al proyectar un documento sobre un STRUCT con campos limitados?
6. Explicá por qué almacenar JSON en Delta no demuestra índices ni garantías de una base documental.